<a href="https://colab.research.google.com/github/D3zNt/Team3AmazonProject/blob/feature%2FYiranQi/try.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# YOLOv8 Training with COCO8 Dataset
# Save this as: notebooks/yolo_training.ipynb
!pip install ultralytics
import os
from pathlib import Path
import yaml
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

import torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12


In [ ]:
import zipfile, tarfile, pathlib, os, binascii
from google.colab import drive
drive.mount('/content/drive')# mount drive
def smart_extract(path, out_dir):
    path = pathlib.Path(path)
    out_dir = pathlib.Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    if not path.exists() or path.stat().st_size < 64:
        raise RuntimeError(f"too small or do not exist: {path} (size={path.stat().st_size if path.exists() else 'NA'})")

    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path, 'r') as zf:
            zf.extractall(out_dir)
        return f"zip accomplished → {out_dir}"

    if tarfile.is_tarfile(path):
        with tarfile.open(path, 'r:*') as tf:
            tf.extractall(out_dir)
        return f"tar accomplished → {out_dir}"

    with open(path,'rb') as f:
        head = f.read(32)
    raise RuntimeError(f"ERROR magic={binascii.hexlify(head)}")

print(smart_extract("/content/drive/MyDrive/mixed_ct.zip", "/content/mixed_ct"))


Mounted at /content/drive
zip accomplished → /content/mixed_ct


In [ ]:
from pathlib import Path
import yaml

ROOT = Path('/content/mixed_ct/mixed_ct')
SPLITS = ['train','val','test']  # 没有就会自动跳过
dy = yaml.safe_load(open(ROOT/'data.yaml'))
NC = dy.get('nc', len(dy.get('names', [])))
print('NC =', NC, '| names =', dy.get('names'))


NC = 2 | names = ['Chair', 'Table']


In [ ]:
from pathlib import Path

def quick_check(root, splits):
    for sp in splits:
        img_dir = root/sp/'images'
        lbl_dir = root/sp/'labels'
        if not img_dir.exists():
            print(f'[{sp}] 跳过'); continue
        imgs = [p for p in img_dir.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}]
        lbls = list(lbl_dir.rglob('*.txt')) if lbl_dir.exists() else []
        img_stems = {p.stem for p in imgs}; lbl_stems = {p.stem for p in lbls}
        miss = img_stems - lbl_stems; orphan = lbl_stems - img_stems
        print(f'[{sp}] images={len(imgs)} labels={len(lbls)} 缺标签={len(miss)} 孤儿标签={len(orphan)}')
        if miss:   print('  缺标签例:', list(miss)[:5])
        if orphan: print('  孤儿标签例:', list(orphan)[:5])

quick_check(ROOT, SPLITS)


[train] images=1108 labels=1108 缺标签=0 孤儿标签=0
[val] images=177 labels=177 缺标签=0 孤儿标签=0
[test] images=179 labels=179 缺标签=0 孤儿标签=0


In [ ]:
import shutil, math

DRY_RUN = False         # ✅ 先预览；确认无误后改成 False 才会覆写
ROUND = 6               # 判定完全重复时的坐标取整精度
MIN_W, MIN_H = 0.002, 0.002   # 极小框阈值（归一化）
IOU_DUP_TH = 0.995      # 同类框 IoU 大于此阈值视为近重复

def iou_xywh(a, b):
    ax1, ay1, ax2, ay2 = a[0]-a[2]/2, a[1]-a[3]/2, a[0]+a[2]/2, a[1]+a[3]/2
    bx1, by1, bx2, by2 = b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
    inter = max(0, min(ax2,bx2)-max(ax1,bx1)) * max(0, min(ay2,by2)-max(ay1,by1))
    area_a = (ax2-ax1)*(ay2-ay1); area_b = (bx2-bx1)*(by2-by1)
    return inter / (area_a + area_b - inter + 1e-12)

def clean_label_lines(lines, nc):
    kept, seen = [], set()
    stats = dict(total=len(lines), kept=0, dup=0, near_dup=0, oob=0, small=0, bad=0)
    # 1) 逐行检查
    for ln in lines:
        ln = ' '.join(ln.strip().split())
        if not ln: continue
        parts = ln.split()
        if len(parts) != 5:
            stats['bad'] += 1; continue
        try:
            cls = int(float(parts[0])); cx,cy,w,h = map(float, parts[1:])
        except:
            stats['bad'] += 1; continue
        if cls < 0 or (nc is not None and cls >= nc):
            stats['bad'] += 1; continue
        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
            stats['oob'] += 1; continue
        if w < MIN_W or h < MIN_H:
            stats['small'] += 1; continue
        key = (cls, round(cx,ROUND), round(cy,ROUND), round(w,ROUND), round(h,ROUND))
        if key in seen:
            stats['dup'] += 1; continue
        seen.add(key)
        kept.append((cls,cx,cy,w,h))
    # 2) 近重复（同类框 IoU 很高）
    final = []
    for cls,cx,cy,w,h in kept:
        is_near_dup = any((c2==cls and iou_xywh((cx,cy,w,h),(x2,y2,w2,h2)) > IOU_DUP_TH) for c2,x2,y2,w2,h2 in final)
        if is_near_dup:
            stats['near_dup'] += 1
        else:
            final.append((cls,cx,cy,w,h))
    stats['kept'] = len(final)
    out_lines = [f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c,x,y,w,h in final]
    return out_lines, stats

summary = dict(files=0, lines=0, kept=0, dup=0, near_dup=0, oob=0, small=0, bad=0, changed=0)
examples = []

for sp in SPLITS:
    lbl_dir = ROOT/sp/'labels'
    if not lbl_dir.exists(): continue
    for txt in sorted(lbl_dir.rglob('*.txt')):
        raw = txt.read_text(encoding='utf-8', errors='ignore').splitlines()
        cleaned, st = clean_label_lines(raw, NC)
        summary['files'] += 1; summary['lines'] += st['total']
        for k in ('kept','dup','near_dup','oob','small','bad'):
            summary[k] += st[k]
        changed = st['total'] != st['kept']
        summary['changed'] += int(changed)
        if changed:
            examples.append((txt, st))
            if not DRY_RUN:
                bak = txt.with_suffix('.txt.bak')
                if not bak.exists(): shutil.copy2(txt, bak)
                txt.write_text('\n'.join(cleaned) + '\n', encoding='utf-8')

print('=== 清洁汇总 ===')
print(summary)
print(f'将被改写的文件数: {summary["changed"]} | DRY_RUN={DRY_RUN}')
for i,(p,st) in enumerate(examples[:10]):
    print(f'{i+1}. {p.name}: 原{st["total"]}→留{st["kept"]}  重复{st["dup"]} 近重复{st["near_dup"]} 越界{st["oob"]} 极小{st["small"]} 非法{st["bad"]}')


=== 清洁汇总 ===
{'files': 1464, 'lines': 8917, 'kept': 4614, 'dup': 4303, 'near_dup': 0, 'oob': 0, 'small': 0, 'bad': 0, 'changed': 1153}
将被改写的文件数: 1153 | DRY_RUN=False
1. 2015_03778.txt: 原4→留2  重复2 近重复0 越界0 极小0 非法0
2. 2015_03779.txt: 原28→留14  重复14 近重复0 越界0 极小0 非法0
3. 2015_03780.txt: 原4→留2  重复2 近重复0 越界0 极小0 非法0
4. 2015_03781.txt: 原20→留10  重复10 近重复0 越界0 极小0 非法0
5. 2015_03782.txt: 原2→留1  重复1 近重复0 越界0 极小0 非法0
6. 2015_03784.txt: 原6→留3  重复3 近重复0 越界0 极小0 非法0
7. 2015_03786.txt: 原14→留7  重复7 近重复0 越界0 极小0 非法0
8. 2015_03787.txt: 原8→留4  重复4 近重复0 越界0 极小0 非法0
9. 2015_03788.txt: 原12→留6  重复6 近重复0 越界0 极小0 非法0
10. 2015_03789.txt: 原2→留1  重复1 近重复0 越界0 极小0 非法0


In [ ]:
import glob, os
for p in glob.glob(str(ROOT/'*'/'labels.cache')):
    try: os.remove(p)
    except FileNotFoundError: pass
print('labels.cache 已清理')


labels.cache 已清理


In [ ]:
from ultralytics import YOLO
import torch
import json
from datetime import datetime
import os

# ==== settings ====
now_str = datetime.now().strftime('%Y%m%d_%H%M%S')
'''config = {
    "model": "yolov8n.pt",
    "data": "/content/mixed_ct/mixed_ct/data.yaml",
    "epochs": 50,# 50
    "imgsz": 768,# 320
    "batch": 32,
    "name": f"furniture_yolov8n_{now_str}",
    "project": "furniture_project_3",
    "exist_ok": True,
    "device": "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.05, # 0
    "copy_paste": 0.0,
    "patience": 50,
    "workers": 8,
    "save": True,
    "save_period": -1,
    "cache": False,
    "close_mosaic": 10,# 0
    "resume": False,
    "amp": True,
    "pretrained": True,
    "erasing": 0.15, #
    "workers" : 8 #


}'''
config = {
    "model": "yolov8n.pt",
    "data": "/content/mixed_ct/mixed_ct/data.yaml",
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "lr0": 0.01,
    "lrf": 0.01,
    "optimizer": "SGD",      # 关键：换回 SGD
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "cos_lr": False,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,

    "mosaic": 1.0,
    "close_mosaic": 10,
    "mixup": 0.05,
    "copy_paste": 0.2,
    "erasing": 0.15,
    "fliplr": 0.5,
    "flipud": 0.0,
    "degrees": 0.0, "translate": 0.1, "scale": 0.5, "shear": 0.0, "perspective": 0.0,

    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,

    "rect": False,
    "patience": 50,
    "workers": 8,
    "amp": True,
    "pretrained": True,
    "project": "furniture_project_3",
    "exist_ok": True,
}

# ==== Train ====
model = YOLO(config["model"])
results = model.train(**config)

# ==== Save ====
save_dir = model.trainer.save_dir
config_path = os.path.join(save_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)
print(f"param saved: {config_path}")

New https://pypi.org/project/ultralytics/8.3.178 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.177 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mixed_ct/mixed_ct/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.15, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimiz

train: Scanning /content/mixed_ct/mixed_ct/train/labels.cache... 1108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1108/1108 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 260.7±139.1 MB/s, size: 84.0 KB)


val: Scanning /content/mixed_ct/mixed_ct/val/labels.cache... 177 images, 0 backgrounds, 0 corrupt: 100%|██████████| 177/177 [00:00<?, ?it/s]


...

      50/50      3.04G     0.9865     0.8589      1.168          6        640: 100%|██████████| 70/70 [00:26<00:00,  2.66it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:01<00:00,  3.38it/s]

                   all        177        477      0.608      0.524      0.573      0.394



50 epochs completed in 0.409 hours.
Optimizer stripped from furniture_project_3/train/weights/last.pt, 6.2MB
Optimizer stripped from furniture_project_3/train/weights/best.pt, 6.2MB

Validating furniture_project_3/train/weights/best.pt...
Ultralytics 8.3.177 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:04<00:00,  1.49it/s]


                   all        177        477       0.65      0.515      0.582      0.401
                 Chair        114        228      0.714      0.583       0.66      0.495
                 Table         63        249      0.585      0.446      0.504      0.307
Speed: 0.3ms preprocess, 2.5ms inference, 0.0ms loss, 7.0ms postprocess per image
Results saved to furniture_project_3/train
param saved: furniture_project_3/train/config.json


In [ ]:
# モデルの読み込み（あなたの学習済み重み）
import matplotlib.pyplot as plt
model = YOLO("/Users/Kota/blended/Team3AmazonProject/notebooks/furniture_project_3/furniture_yolov8n_20250805_223347/weights/best.pt")

# 推論したい画像パス
image_path = "/Users/Kota/blended/Team3AmazonProject/data/temp/original/Chair--2-_jpg.rf.53703002ffac527a3c2d19b775620bd2.jpg"  # ← 任意の画像パスに置き換えてください

# 推論
results = model(image_path)

# 結果の表示（画像をプロット）
results[0].show()  # OpenCVウィンドウで表示（環境によっては使えない場合あり）

# matplotlibで表示（macなどGUI非対応環境向け）
img = results[0].plot()  # 結果画像を取得（NumPy array）
plt.imshow(img)
plt.axis('off')
plt.show()


image 1/1 /Users/Kota/blended/Team3AmazonProject/data/temp/original/Chair--2-_jpg.rf.53703002ffac527a3c2d19b775620bd2.jpg: 320x320 1 Chair, 16.1ms
Speed: 0.7ms preprocess, 16.1ms inference, 0.4ms postprocess per image at shape (1, 3, 320, 320)


<Figure size 640x480 with 1 Axes>

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import os
from pathlib import Path

# Input and output directories
input_dir = Path("/Users/Kota/blended/Team3AmazonProject/data/temp/original")
output_dir = Path("/Users/Kota/blended/Team3AmazonProject/data/temp/cropped")
output_dir.mkdir(parents=True, exist_ok=True)

# Load YOLOv8 segmentation model
model = YOLO("yolov8x-seg.pt")

# Supported image extensions
image_exts = [".jpg", ".jpeg", ".png"]

# Iterate over all images in the input directory
for img_path in input_dir.glob("*"):
    if img_path.suffix.lower() not in image_exts:
        continue

    # Read the image
    original = cv2.imread(str(img_path))
    if original is None:
        print(f"⚠️ Failed to read: {img_path}")
        continue

    height, width = original.shape[:2]

    # Run inference without resizing
    results = model(source=str(img_path), imgsz=(width, height))[0]

    # Skip if no masks detected
    if results.masks is None or len(results.masks.data) == 0:
        print(f"❌ No mask detected: {img_path.name}")
        continue

    # Use only the first detected object
    mask = results.masks.data[0].cpu().numpy().astype(np.uint8)

    # Resize the mask to match the original image size
    mask_resized = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)

    # Apply the mask to the entire image (black background outside object)
    mask_3c = np.stack([mask_resized] * 3, axis=-1)
    masked_img = original * mask_3c  # Same shape as original

    # Save masked image (same size as original)
    base_name = img_path.stem
    save_path = output_dir / f"{base_name}_cropped.png"
    cv2.imwrite(str(save_path), masked_img)
    print(f"✅ Saved: {save_path}")

100%|██████████| 137M/137M [00:14<00:00, 9.88MB/s] 



WARNING ⚠️ imgsz=[1200, 1200] must be multiple of max stride 32, updating to [1216, 1216]
image 1/1 /Users/Kota/blended/Team3AmazonProject/data/temp/original/Table--154-_jpg.rf.67264c7a7b156dd01c44e5e35de6cbe9.jpg: 1216x1216 (no detections), 2076.0ms
Speed: 7.4ms preprocess, 2076.0ms inference, 2.3ms postprocess per image at shape (1, 3, 1216, 1216)
❌ No mask detected: Table--154-_jpg.rf.67264c7a7b156dd01c44e5e35de6cbe9.jpg

WARNING ⚠️ imgsz=[1200, 1200] must be multiple of max stride 32, updating to [1216, 1216]
image 1/1 /Users/Kota/blended/Team3AmazonProject/data/temp/original/Sofa--348-_jpg.rf.74a1bda29972fc468b85d6c9eab48bbd.jpg: 1216x1216 2 couchs, 2024.8ms
Speed: 4.4ms preprocess, 2024.8ms inference, 6.1ms postprocess per image at shape (1, 3, 1216, 1216)
✅ Saved: /Users/Kota/blended/Team3AmazonProject/data/temp/cropped/Sofa--348-_jpg.rf.74a1bda29972fc468b85d6c9eab48bbd_cropped.png

WARNING ⚠️ imgsz=[1200, 1200] must be multiple of max stride 32, updating to [1216, 1216]
image 